In [14]:
# ── 1. Config & Imports ────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# File paths
receipts_orc_path = "data/receipts_orc.xlsx"
rests_orc_path    = "data/rests_orc.txt"
rests_tt_path     = "data/rests_tt.txt"
sales_path        = "data/sales.txt"
shipment_path     = "data/shipment.txt"

# ABT period range
PERIOD_START = '2020-01'
PERIOD_END   = '2026-02'


In [16]:
# ── 2. Ingestion & Standardization ─────────────────────────────────────────

# Column schemas
columns_orc   = ['Дата', 'Артикул', 'Количество', 'Стоимость']
columns_tt    = ['Дата', 'Партнер', 'Артикул', 'Количество', 'Стоимость']
columns_sales = ['Партнер', 'Артикул', 'Дата', 'Количество', 'Выручка']

dtype_sku       = {'Артикул': str, 'Партнер': str}
num_names       = ['Количество', 'Выручка']
num_names_rests = ['Количество', 'Стоимость']


def read_txt(
    path: str,
    names: list[str],
    dtype: dict,
    encoding: str = 'cp1251',
    skiprows: int = 5,
    drop_subset: list[str] = ['Артикул']
) -> pd.DataFrame:
    # Force numeric 1C columns to str so pandas never infers mixed types;
    # clean_numeric() in standardize() handles the actual conversion.
    full_dtype = {'Количество': str, 'Стоимость': str, 'Выручка': str, **dtype}
    df = pd.read_csv(
        path,
        sep='\t',
        skiprows=skiprows,
        names=names,
        encoding=encoding,
        decimal=',',
        dtype=full_dtype,
        on_bad_lines='warn'
    )
    return df.dropna(subset=drop_subset)


def clean_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str).str.replace(r'\s+', '', regex=True).str.replace(',', '.'),
        errors='coerce'
    ).fillna(0.0)


def standardize(df: pd.DataFrame, numeric_names: list[str]) -> pd.DataFrame:
    """Strip key columns, cast numerics, convert Дата → Период, drop Дата."""
    if 'Артикул' in df.columns:
        df['Артикул'] = df['Артикул'].astype(str).str.strip()
    if 'Партнер' in df.columns:
        df['Партнер'] = df['Партнер'].astype(str).str.strip()
    for col in numeric_names:
        if col in df.columns:
            df[col] = clean_numeric(df[col])
    if 'Дата' in df.columns:
        df['Период'] = pd.to_datetime(df['Дата'], format='%d.%m.%Y', errors='coerce').dt.to_period('M')
        df = df.drop(columns=['Дата'])
    return df


# .txt sources (cp1251 default for ORC/TT, utf-8 for sales/shipment)
rests_orc_df = standardize(
    read_txt(rests_orc_path, columns_orc, dtype_sku),
    num_names_rests
)
rests_tt_df = standardize(
    read_txt(rests_tt_path, columns_tt, dtype_sku),
    num_names_rests
)
sales_df = standardize(
    read_txt(sales_path, columns_sales, dtype_sku, encoding='utf-8', skiprows=6),
    num_names
)
shipment_df = standardize(
    read_txt(shipment_path, columns_sales, dtype_sku, encoding='utf-8', skiprows=6),
    num_names
)

# .xlsx source — Дата is already datetime from Excel, handle separately
receipts_orc_df = pd.read_excel(receipts_orc_path)
receipts_orc_df['Артикул']  = receipts_orc_df['Артикул'].astype(str).str.strip()
receipts_orc_df['Период']   = receipts_orc_df['Дата'].dt.to_period('M')

receipts_orc_df['Количество'] = receipts_orc_df['Количество'].astype('float64')
receipts_orc_df = receipts_orc_df.drop(columns=['Дата'])

In [18]:
# ── 3. Monthly Aggregation ──────────────────────────────────────────────────
# All transactional frames MUST be aggregated to monthly grain before any join.

# Partner-level grain (joined later on ['Период', 'Партнер', 'Артикул'])
sales_agg = (
    sales_df
    .groupby(['Период', 'Партнер', 'Артикул'], as_index=False)
    .agg({'Количество': 'sum', 'Выручка': 'sum'})
    .rename(columns={'Количество': 'Количество_sales', 'Выручка': 'Выручка_sales'})
)

shipment_agg = (
    shipment_df
    .groupby(['Период', 'Партнер', 'Артикул'], as_index=False)
    .agg({'Количество': 'sum', 'Выручка': 'sum'})
    .rename(columns={'Количество': 'Количество_ship', 'Выручка': 'Выручка_ship'})
)

rests_tt_agg = (
    rests_tt_df
    .groupby(['Период', 'Партнер', 'Артикул'], as_index=False)
    .agg({'Количество': 'sum', 'Стоимость': 'sum'})
)
rests_tt_agg['Количество'] = rests_tt_agg['Количество'].clip(lower=0)
rests_tt_agg = rests_tt_agg.rename(columns={'Количество': 'Количество_tt', 'Стоимость': 'Стоимость_tt'})

# Warehouse grain (joined later on ['Период', 'Артикул'] — broadcasts across all partners)
rests_orc_agg = (
    rests_orc_df
    .groupby(['Период', 'Артикул'], as_index=False)
    .agg({'Количество': 'sum', 'Стоимость': 'sum'})
)
rests_orc_agg['Количество'] = rests_orc_agg['Количество'].clip(lower=0)
rests_orc_agg = rests_orc_agg.rename(columns={'Количество': 'Количество_orc', 'Стоимость': 'Стоимость_orc'})

receipts_orc_agg = (
    receipts_orc_df
    .groupby(['Период', 'Артикул'], as_index=False)
    .agg({'Количество': 'sum', 'ЦенаВВалюте': 'mean'})
    .rename(columns={'Количество': 'Количество_receipts'})
)


In [19]:
# ── 4. Skeleton Construction & Left Joins ───────────────────────────────────

# Unique Partner × SKU pairs from all historical partner-level sources
pairs_cols   = ['Партнер', 'Артикул']
unique_pairs = pd.concat([
    sales_agg[pairs_cols],
    shipment_agg[pairs_cols],
    rests_tt_agg[pairs_cols],
]).drop_duplicates().reset_index(drop=True)

# Cross pairs with full period range → dense skeleton
all_periods = pd.period_range(start=PERIOD_START, end=PERIOD_END, freq='M')

index = pd.MultiIndex.from_product(
    [all_periods, unique_pairs.index],
    names=['Период', 'pair_index']
)
skeleton_df = (
    pd.DataFrame(index=index)
    .reset_index()
    .merge(unique_pairs, left_on='pair_index', right_index=True)
    .drop(columns=['pair_index'])
)

# Sequential left joins — skeleton is always the left table
df_master = (
    skeleton_df
    .merge(sales_agg,        on=['Период', 'Партнер', 'Артикул'], how='left')
    .merge(shipment_agg,     on=['Период', 'Партнер', 'Артикул'], how='left')
    .merge(rests_tt_agg,     on=['Период', 'Партнер', 'Артикул'], how='left')
    .merge(rests_orc_agg,    on=['Период', 'Артикул'],            how='left')
    .merge(receipts_orc_agg, on=['Период', 'Артикул'],            how='left')
)

# NaN from left join = absent transaction = physical zero
metric_cols = [c for c in df_master.columns if c not in ('Период', 'Партнер', 'Артикул')]
df_master[metric_cols] = df_master[metric_cols].fillna(0.0)

df_master.head()


,Период,Партнер,Артикул,Количество_sales,Выручка_sales,Количество_ship,Выручка_ship,Количество_tt,Стоимость_tt,Количество_orc,Стоимость_orc,Количество_receipts,ЦенаВВалюте
0,2020-01,Євростар,DJ01654,1.0,399.36,0.0,0.0,0.0,0.0,19.0,6503.70,0.0,0.0
1,2020-01,Євростар,DJ03085,1.0,466.02,0.0,0.0,0.0,0.0,15.0,5991.30,0.0,0.0
2,2020-01,Євростар,DJ07112,1.0,406.02,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0
3,2020-01,Євростар,DJ08320,2.0,505.44,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0
4,2020-01,Євростар,DJ08652,1.0,466.02,0.0,0.0,0.0,0.0,21.0,8387.82,0.0,0.0


In [11]:
# ── Verification ────────────────────────────────────────────────────────────
assert df_master.shape[0] == len(all_periods) * len(unique_pairs), "Row count mismatch: skeleton is not fully dense"
assert df_master[metric_cols].isna().sum().sum() == 0, "Unexpected NaN in metric columns after fillna"
assert rests_orc_agg['Количество_orc'].min() >= 0, "Negative ORC inventory after clip"
assert rests_tt_agg['Количество_tt'].min() >= 0, "Negative TT inventory after clip"

print(f"df_master shape : {df_master.shape}")
print(f"Periods         : {len(all_periods)}")
print(f"Unique pairs    : {len(unique_pairs)}")
print(f"Columns         : {list(df_master.columns)}")
df_master.dtypes


df_master shape : (2859730, 13)
Periods         : 74
Unique pairs    : 38645
Columns         : ['Период', 'Партнер', 'Артикул', 'Количество_sales', 'Выручка_sales', 'Количество_ship', 'Выручка_ship', 'Количество_tt', 'Стоимость_tt', 'Количество_orc', 'Стоимость_orc', 'Количество_receipts', 'ЦенаВВалюте']


Период                 period[M]
Партнер                      str
Артикул                      str
Количество_sales         float64
Выручка_sales            float64
Количество_ship          float64
Выручка_ship             float64
Количество_tt            float64
Стоимость_tt             float64
Количество_orc           float64
Стоимость_orc            float64
Количество_receipts      float64
ЦенаВВалюте              float64
dtype: object